# NetworkX: Network Properties & Centrality Analysis

---

In Notebook 1, you learned how to build graphs from scratch: adding nodes and edges, attaching attributes (like weights), choosing between graph types (`Graph`, `DiGraph`, `MultiDiGraph`), and drawing basic layouts.

Now, we shift from **building graphs** to **analyzing them**. When looking at a real-world network (like a social network, a citation graph, or a knowledge graph), we usually want to answer two big questions:
1. **How is the overall network structured?** (Macroscopic / Global Properties)
2. **Which individual nodes are the most important?** (Microscopic / Centrality Measures)

### What You'll Learn in This Notebook:
1. **Setup & Environment:** Preparing our Python libraries (`networkx`, `matplotlib`, `numpy`, and `pandas`).
2. **Global Network Metrics:** Measuring how dense, connected, and spread out a network is.
3. **Handling Disconnected Graphs:** Avoiding common beginner errors when a graph breaks into separate pieces.
4. **Node Centrality (Undirected):** Finding "popular" nodes, "fast spreaders," and "bridge/gatekeeper" nodes.
5. **Weighted vs. Unweighted Centrality:** Understanding how edge weights change NetworkX calculations.
6. **Directed Influence & PageRank:** Measuring authority and popularity when connections have a direction.

### 1. Setup & Environment
First, let's import all the libraries we need right at the top of the notebook:
* `networkx` (`nx`) for graph creation and algorithms.
* `matplotlib.pyplot` (`plt`) for drawing our graphs.
* `numpy` (`np`) for numerical operations.
* `pandas` (`pd`) for organizing our node metrics into clean, readable tables.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Configure visual styling for crisp, readable plots
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

### 2. Macroscopic (Global) Network Properties
Before zooming in on individual nodes, we always look at the "big picture." Global properties summarize the overall structure, tightness, and communication efficiency of an entire network.

Let's define the most important global metrics:

* **Density ($0.0$ to $1.0$):** How tightly knit is the network? It compares the number of edges that *actually exist* ($E$) to the *maximum possible edges* if everyone were connected to everyone else. For an undirected graph with $N$ nodes:
  $$\text{Density} = \frac{2E}{N(N-1)}$$
* **Connected Components:** How many separate "islands" (disconnected subgraphs) exist?
* **Shortest Path (Geodesic Distance):** The minimum number of steps (hops) required to travel from Node A to Node B.
* **Diameter:** It tells you how many steps it takes to cross from one extreme end of the network to the other.
* **Average Shortest Path Length:** On average, how many steps does it take to get from one random person in the network to another? 

Let's build a small **Team Collaboration Network** where edge `weight` represents how many projects two people have worked on together, and draw a quick preview first so we can see what we're measuring!


In [ ]:
# Create an undirected collaboration network
G = nx.Graph()

# Each tuple is (Person A, Person B, Number of shared projects)
edges = [
    ("Alice", "Bob", 4),
    ("Alice", "Charlie", 2),
    ("Bob", "Charlie", 1),
    ("Bob", "Diana", 5),
    ("Charlie", "Diana", 2),
    ("Charlie", "Eve", 3),
    ("Diana", "Eve", 2),
    ("Diana", "Frank", 6),
    ("Eve", "Frank", 3),
    ("Frank", "Gita", 4),
    ("Gita", "Hank", 2)
]

# Add weighted edges to the graph
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# Quick visual preview of our network topology
plt.figure(figsize=(8, 4))
pos_preview = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos_preview, node_color="#A0CBE2", node_size=900, edgecolors="black", font_weight="bold")
edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos_preview, edge_labels=edge_labels, font_color="darkred")
plt.title("Team Collaboration Network (Edge Labels = Shared Projects)", fontweight="bold")
plt.axis("off")
plt.show()


Now let's calculate the global metrics for our collaboration network `G`:

In [ ]:
# 1. Network Density
density = nx.density(G)

# 2. Connectivity & Components
is_connected = nx.is_connected(G)
num_components = nx.number_connected_components(G)

# 3. Path Metrics (Note: Only valid if the graph is fully connected!)
if is_connected:
    diameter = nx.diameter(G)
    avg_path_length = nx.average_shortest_path_length(G)
else:
    diameter = float("inf")
    avg_path_length = float("inf")

print(f"Total Nodes:                  {G.number_of_nodes()}")
print(f"Total Edges:                  {G.number_of_edges()}")
print(f"Network Density:              {density:.4f} (about 39.3% of all possible connections exist)")
print(f"Is Fully Connected?:          {is_connected} ({num_components} connected component)")
print(f"Network Diameter:             {diameter} hops (e.g., Alice -> Bob -> Diana -> Frank -> Gita -> Hank)")
print(f"Average Shortest Path Length: {avg_path_length:.4f} hops")


### 3. Common Pitfall: Handling Disconnected Graphs
What happens if your network has an isolated node or breaks into two separate groups? 

If there is no path between two nodes, the distance between them is **infinite ($\infty$)**. Because of this, calling `nx.diameter(G)` or `nx.average_shortest_path_length(G)` on a disconnected graph will immediately crash with a `NetworkXError: Found infinite path length`.

**Best Practice:** In real-world projects, always check `nx.is_connected(G)` first, or extract the **Largest Connected Component (LCC)**—also called the "Giant Component"—and measure path lengths on that!


In [ ]:
# Let's create a copy of G and add an isolated 2-person team
G_disconnected = G.copy()
G_disconnected.add_edge("Isolate_1", "Isolate_2", weight=1)

print(f"Is G_disconnected connected?: {nx.is_connected(G_disconnected)}")

# 1. See what happens if we try to compute diameter directly (caught safely with try/except)
try:
    nx.diameter(G_disconnected)
except nx.NetworkXError as e:
    print(f"Caught Expected Error: {e}\n")

# 2. Extract all connected components (returns sets of node names)
components = list(nx.connected_components(G_disconnected))
print(f"Total separate subgraphs (components): {len(components)}")
for idx, comp in enumerate(components, 1):
    print(f"  Component {idx}: {comp}")

# 3. Isolate the Largest Connected Component (LCC) as its own Graph object
lcc_nodes = max(nx.connected_components(G_disconnected), key=len)
LCC = G_disconnected.subgraph(lcc_nodes).copy()

print(f"\nNodes in Largest Connected Component (LCC): {list(LCC.nodes())}")
print(f"Diameter of LCC: {nx.diameter(LCC)} hops")


### 4. Node-Level Importance: Centrality Measures
Now let's zoom in from the whole network to individual nodes. **Centrality** answers the question: *"Which node is the most important in this network?"*

Because "important" can mean different things depending on your goal, NetworkX provides four classic centrality algorithms:

| Centrality Metric | Real-World Intuition | How It's Calculated | NetworkX Function |
| :--- | :--- | :--- | :--- |
| **Degree Centrality** | **Local Popularity:** Who has the most direct connections? | Number of neighbors divided by maximum possible neighbors ($\frac{\text{deg}(u)}{N-1}$). | `nx.degree_centrality(G)` |
| **Closeness Centrality** | **Fast Spreader:** Who can reach everyone else in the fewest steps? | $1$ divided by the average shortest path distance from node $u$ to all other nodes. | `nx.closeness_centrality(G)` |
| **Betweenness Centrality** | **Bridge / Gatekeeper:** Who connects different groups and controls information flow? | Fraction of all shortest paths between *all pairs of nodes* that pass through node $u$. | `nx.betweenness_centrality(G)` |
| **Eigenvector Centrality** | **VIP Influence:** Who is connected to *other* highly connected people? | A node's score is proportional to the sum of the scores of its neighbors. | `nx.eigenvector_centrality(G)` |

Let's compute all four metrics on our team network `G` and compare them side-by-side in a Pandas DataFrame:


In [ ]:
# Compute the 4 standard centrality metrics (each returns a dictionary of {node: score})
deg_centrality = nx.degree_centrality(G)
closeness_centrality = nx.closeness_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
eigen_centrality = nx.eigenvector_centrality(G, max_iter=500)

# Combine them into a clean Pandas DataFrame for easy comparison
centrality_df = pd.DataFrame({
    "Degree (Popularity)": deg_centrality,
    "Closeness (Reach Speed)": closeness_centrality,
    "Betweenness (Bridge)": betweenness_centrality,
    "Eigenvector (Prestige)": eigen_centrality
}).round(4)

# Sort by Betweenness Centrality from highest to lowest
centrality_df.sort_values(by="Betweenness (Bridge)", ascending=False)


#### How to Read This Table (Crucial Insight!):
Compare **Diana / Charlie** with **Frank**:
* **Diana and Charlie** have the highest **Degree Centrality ($0.5714$)** and **Eigenvector Centrality**. They sit right in the middle of the busy team cluster (`Alice`, `Bob`, `Charlie`, `Diana`, `Eve`).
* **Frank**, however, has the highest **Betweenness Centrality ($0.4762$)** by far! Why? Look back at our graph drawing in Section 2: Frank is the **only bridge** connecting the main team to `Gita` and `Hank`. Every shortest path between the main team and `Gita`/`Hank` *must* pass through Frank. If Frank leaves the company, `Gita` and `Hank` are completely cut off!


---
### 5. How Do Edge Weights Affect Centrality?
Earlier, we added `weight` attributes to our edges (representing the number of shared projects). 

By default, `nx.degree_centrality(G)`, `nx.closeness_centrality(G)`, and `nx.betweenness_centrality(G)` **ignore edge weights** unless you explicitly tell them to use weights!
* If you want total weighted connections (often called **Node Strength**), use `G.degree(weight="weight")`.
* In path-based metrics (`closeness` and `betweenness`), NetworkX treats edge weights as **distance/cost** (where a *larger* number means *farther apart*), NOT connection strength! If a higher weight means a *stronger/closer* bond (like shared projects), you must invert the weight ($\text{distance} = \frac{1}{\text{weight}}$) before running weighted path algorithms.


In [ ]:
# Compare Unweighted Degree (number of collaborators) vs Weighted Strength (total shared projects)
unweighted_deg = dict(G.degree())
weighted_deg = dict(G.degree(weight="weight"))

weight_df = pd.DataFrame({
    "Unweighted Degree (Collaborators)": unweighted_deg,
    "Weighted Strength (Total Projects)": weighted_deg
}).sort_values(by="Weighted Strength (Total Projects)", ascending=False)

weight_df


### 6. Directed Influence: In-Degree, Out-Degree, and PageRank

**Directed Graph (`nx.DiGraph`)**

1. **In-Degree Centrality (`nx.in_degree_centrality`):** How many arrows point INTO this node?
2. **Out-Degree Centrality (`nx.out_degree_centrality`):** How many arrows point OUT from this node?

#### What is PageRank (`nx.pagerank`)?
Originally invented by Google's founders to rank web pages, PageRank takes In-Degree a step further. It asks: *"Do you just have a lot of incoming arrows, or do you have incoming arrows from other highly important nodes?

* Getting $1$ incoming arrow from a major hub node (like `Large Language Model`) boosts your **PageRank** much more than getting $1$ arrow from a peripheral node!

Let's build a **Directed AI Knowledge Graph**:


In [ ]:
# Build an applied Directed Knowledge / Dependency Network
dependency_graph = nx.DiGraph()

edges_dir = [
    ("Neural Network", "Deep Learning"),
    ("Gradient Descent", "Deep Learning"),
    ("Backpropagation", "Gradient Descent"),
    ("Deep Learning", "Transformer"),
    ("Attention Mechanism", "Transformer"),
    ("Transformer", "Large Language Model"),
    ("Large Language Model", "GraphRAG"),
    ("Knowledge Graph", "GraphRAG"),
    ("Vector DB", "GraphRAG")
]

dependency_graph.add_edges_from(edges_dir)

# Visualize the Directed Graph so we can trace the arrows!
plt.figure(figsize=(10, 4.5))
pos_dir = nx.spring_layout(dependency_graph, seed=10, k=0.9)
nx.draw_networkx(
    dependency_graph, pos_dir,
    node_color="#FFD580", node_size=1200, edgecolors="black",
    font_size=8, font_weight="bold",
    arrows=True, arrowsize=18, arrowstyle="-|>", edge_color="#555555"
)
plt.title("Directed AI Knowledge Dependency Graph", fontweight="bold")
plt.axis("off")
plt.show()

# 1. Compute In-Degree Centrality vs Out-Degree Centrality
in_deg = nx.in_degree_centrality(dependency_graph)
out_deg = nx.out_degree_centrality(dependency_graph)

# 2. Compute PageRank (damping factor alpha=0.85)
pagerank_scores = nx.pagerank(dependency_graph, alpha=0.85)

dir_df = pd.DataFrame({
    "In-Degree (Popularity)": in_deg,
    "Out-Degree (Broadcast)": out_deg,
    "PageRank (Authority)": pagerank_scores
}).round(4)

dir_df.sort_values(by="PageRank (Authority)", ascending=False)


#### Look at `Large Language Model` vs. `Deep Learning`:
* `Deep Learning` has **more incoming arrows** (`In-Degree` $= 0.2222$, from `Neural Network` and `Gradient Descent`) than `Large Language Model` (`In-Degree` $= 0.1111$, only from `Transformer`).
* Yet `Large Language Model` has a **higher PageRank ($0.1864$ vs. $0.1265$)**! Why? Because `Large Language Model` receives all the accumulated flow coming down the entire chain (`Backpropagation` $\rightarrow$ `Gradient Descent` $\rightarrow$ `Deep Learning` $\rightarrow$ `Transformer` $\rightarrow$ `Large Language Model`)!


---
### 7. Exercises

Test what you've learned by modifying the directed graphs and observing how the network metrics change!

#### Exercise 1: Boosting PageRank in a Directed Graph
In Section 6, `GraphRAG` had the highest PageRank in `dependency_graph` because three arrows pointed into it, and it had zero outgoing arrows.
1. Create a copy of `dependency_graph` called `DG_updated` and add a directed edge from `"GraphRAG"` to a new node called `"AI Agent"`.
2. Compute the updated **PageRank** (`nx.pagerank`), **In-Degree** (`nx.in_degree_centrality`), and **Out-Degree** (`nx.out_degree_centrality`) for `DG_updated`.
3. Visualize the updated graph and display the full table of scores sorted by PageRank!

In [ ]:
# Exercise 1: Complete Solution with Graph & Full Table
DG_updated = dependency_graph.copy()

# 1. Add a directed edge from "GraphRAG" to "AI Agent"
DG_updated.add_edge("GraphRAG", "AI Agent")

# 2. Compute In-Degree, Out-Degree, and PageRank on DG_updated
updated_in_deg = nx.in_degree_centrality(DG_updated)
updated_out_deg = nx.out_degree_centrality(DG_updated)
updated_pr = nx.pagerank(DG_updated)

# 3. Visualize the updated Directed Graph
plt.figure(figsize=(10, 4.5))
pos_updated = nx.spring_layout(DG_updated, seed=10, k=0.9)
nx.draw_networkx(
    DG_updated, pos_updated,
    node_color="#B5EAD7", node_size=1200, edgecolors="black",
    font_size=8, font_weight="bold",
    arrows=True, arrowsize=18, arrowstyle="-|>", edge_color="#555555"
)
plt.title("Updated AI Knowledge Dependency Graph (with AI Agent)", fontweight="bold")
plt.axis("off")
plt.show()

# 4. Print the top node and display the full DataFrame of scores
top_node = max(updated_pr, key=updated_pr.get)
print(f"Top Authority Node: {top_node} (PageRank = {updated_pr[top_node]:.4f})\n")

updated_df = pd.DataFrame({
    "In-Degree (Popularity)": updated_in_deg,
    "Out-Degree (Broadcast)": updated_out_deg,
    "PageRank (Authority)": updated_pr
}).round(4)

updated_df.sort_values(by="PageRank (Authority)", ascending=False)